# Session 7 – Artificial Neural Networks: Student Tasks

**Module:** CPU5006-20 Artificial Intelligence
**Session:** 7 of 12
**Student name:** ____________________
**Date:** ____________________

## How to use this notebook
This notebook goes with the **Session 7: Artificial Neural Networks** slides, and follows them in order. It has two kinds of cell:

- **Warm-up** cells are the code examples from the slides, in the order they appear. They are complete and working — run them as we go through the session. Each one is headed with the title of its slide.
- **Task** cells are for you to complete. Starter code and `# TODO` comments show you where to work, and each task has **hints** you can open one at a time if you get stuck.

Every model is shown in **Keras** and in **PyTorch**. If TensorFlow won't install on your laptop, skip the Keras cells and do everything in PyTorch.

### Learning outcomes
By the end of this session, you should be able to:
- explain the **Perceptron**, why it cannot solve XOR, and how the **MLP** fixes this
- explain, at a high level, what **backpropagation** does
- build, train and evaluate an **MLP classifier** and an **MLP regressor**, in **Keras** *and* **PyTorch**
- choose the right **output activation** and **loss function** for a classification or regression problem
- evaluate on the **test set**, and spot **over-fitting** in the learning curves

### Setup check — run this first
If `tensorflow` shows as MISSING and `pip install tensorflow` doesn't work on your laptop, skip the **Keras** cells and use **PyTorch**. Everything else still runs.

In [ ]:
import importlib

for name in ["numpy", "pandas", "matplotlib", "sklearn", "torch", "torchvision", "tensorflow"]:
    try:
        module = importlib.import_module(name)
        print(f"OK       {name} {getattr(module, '__version__', '')}")
    except ImportError:
        print(f"MISSING  {name}")

---
# Part 1 – A Brief History of Neural Networks

## Activity: Verify the XOR Network  ⏱ 5 min · in pairs
Slide: *Activity: Verify the XOR Network*. Work out the table on paper first, then check your answers here. The two hidden neurons are done for you; complete the output neuron.

### Hints
Try the task first. Open one hint at a time if you get stuck.

<details><summary><b>Hint 1:</b> where are the output neuron's weights?</summary>

On the slide: the output neuron takes **−1 × h₁**, **+1 × h₂**, and a bias of **−0.5**, then applies `step`.

</details>

<details><summary><b>Hint 2:</b> what should the four outputs be?</summary>

XOR is 1 when exactly one input is 1: (0,0)→0, (0,1)→1, (1,0)→1, (1,1)→0. If your `y` column matches, you're done.

</details>

In [ ]:
def step(z):
    return 1 if z >= 0 else 0


def xor_mlp(x1, x2):
    h1 = step(x1 + x2 - 1.5)
    h2 = step(x1 + x2 - 0.5)
    # TODO: compute the output y from h1 and h2, using the weights on the slide
    y = ...
    return h1, h2, y


# TODO: uncomment to check all four inputs
# for x1, x2 in [(0, 0), (0, 1), (1, 0), (1, 1)]:
#     print((x1, x2), "->", xor_mlp(x1, x2))

---
# Part 2 – Building a Classifier (Fashion MNIST)

### Warm-up: *Building an Image Classifier: load the data*
If TensorFlow isn't installed, the same dataset loads through `torchvision` instead.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from tensorflow import keras
    (X_train_full, y_train_full), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()
    print("tf is activated")
except ImportError:
    from torchvision import datasets
    train_set = datasets.FashionMNIST("data", train=True, download=True)
    test_set = datasets.FashionMNIST("data", train=False, download=True)
    X_train_full, y_train_full = train_set.data.numpy(), train_set.targets.numpy()
    X_test, y_test = test_set.data.numpy(), test_set.targets.numpy()
    print("pyTorch is activated")

print(X_train_full.shape)

X_valid, X_train = X_train_full[:5000] / 255., X_train_full[5000:] / 255.
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]
X_test = X_test / 255.

plt.imshow(X_train[0], cmap="binary")
plt.axis('off')
plt.show()

### Warm-up: *Fashion MNIST: the ten classes*

In [ ]:
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

print(class_names[y_train[0]])

n_rows, n_cols = 4, 10
plt.figure(figsize=(n_cols * 1.2, n_rows * 1.2))
for row in range(n_rows):
    for col in range(n_cols):
        index = n_cols * row + col
        plt.subplot(n_rows, n_cols, index + 1)
        plt.imshow(X_train[index], cmap="binary", interpolation="nearest")
        plt.axis('off')
        plt.title(class_names[y_train[index]], fontsize=12)
plt.subplots_adjust(wspace=0.2, hspace=0.5)
plt.show()

### Warm-up: *Creating the ANN with Keras* (Keras)
Two ways to build the same network. The second one replaces the first.

In [ ]:
import tensorflow as tf
from tensorflow import keras

# Method 1: add() one layer at a time
model = keras.models.Sequential()
model.add(keras.Input(shape=[28, 28]))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(300, activation="relu"))
model.add(keras.layers.Dense(100, activation="relu"))
model.add(keras.layers.Dense(10, activation="softmax"))

# Method 2: pass a list of layers to Sequential()
model = keras.models.Sequential([
    keras.Input(shape=[28, 28]),
    keras.layers.Flatten(),
    keras.layers.Dense(300, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])

### Warm-up: *Checking the Architecture* (Keras)

In [ ]:
model.summary()

### Warm-up: *Compile the Model* (Keras)

In [ ]:
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["acc"],
)

### Warm-up: *Train the Model* (Keras)
The slide trains for 30 epochs. 10 is enough to see the pattern, and takes about a minute.

In [ ]:
history = model.fit(X_train, y_train, epochs=10,
                    validation_data=(X_valid, y_valid))

### Warm-up: *Visualise Results* (Keras)

In [ ]:
pd.DataFrame(history.history).plot(figsize=(8, 5))
plt.grid(True)
plt.gca().set_ylim(0, 1)
plt.show()

### Warm-up: *Is It Over-fitting?* (Keras)
This continues training the model above, so `EarlyStopping` usually stops after a few epochs.

In [ ]:
# score the test set once, at the very end
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_acc:.3f}")

# or let validation loss decide when to stop training
early_stop = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=100,
                    validation_data=(X_valid, y_valid),
                    callbacks=[early_stop])
print("Stopped after", len(history.history["loss"]), "more epochs")

### Warm-up: *Making Predictions* (Keras)

In [ ]:
X_new = X_test[:3]
y_proba = model.predict(X_new)
print(y_proba.round(2))

y_pred = np.argmax(model.predict(X_new), axis=-1)
print(np.array(class_names)[y_pred])

---
## The Same Classifier in PyTorch
Same model, same data. The difference is that you write the training loop yourself, where Keras hides it inside `.fit()`.

### Warm-up: *PyTorch: define the model*

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = (
    "cuda" if torch.cuda.is_available()              # NVIDIA GPU
    else "mps" if torch.backends.mps.is_available()  # Apple Silicon Macs
    else "cpu"
)
print("Using:", device)

model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28 * 28, 300), nn.ReLU(),
    nn.Linear(300, 100), nn.ReLU(),
    nn.Linear(100, 10),          # raw scores ("logits") -- no softmax here
).to(device)

loss_fn = nn.CrossEntropyLoss()  # applies softmax + cross-entropy together
optimizer = torch.optim.Adam(model.parameters())

### Warm-up: *PyTorch: the training loop*
The slide trains for 30 epochs; 10 here. The validation check at the end of each epoch is extra, so you can compare with the Keras output.

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
train_loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                           batch_size=32, shuffle=True)

X_valid_t = torch.tensor(X_valid, dtype=torch.float32).to(device)
y_valid_t = torch.tensor(y_valid, dtype=torch.long).to(device)

for epoch in range(10):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()           # clear old gradients
        loss = loss_fn(model(xb), yb)   # forward pass
        loss.backward()                 # backward pass
        optimizer.step()                # gradient descent step

    model.eval()
    with torch.no_grad():
        val_acc = (model(X_valid_t).argmax(1) == y_valid_t).float().mean()
    print(f"epoch {epoch + 1}: validation accuracy {val_acc:.3f}")

### Warm-up: *PyTorch: making predictions*

In [ ]:
model.eval()
with torch.no_grad():
    X_new_t = torch.tensor(X_test[:3], dtype=torch.float32).to(device)
    y_proba = torch.softmax(model(X_new_t), dim=1)
    y_pred = y_proba.argmax(dim=1)

print(y_pred)
print(np.array(class_names)[y_pred.cpu().numpy()])

# and the whole test set, to compare with Keras
with torch.no_grad():
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)
    test_acc = (model(X_test_t).argmax(1) == y_test_t).float().mean()
print(f"Test accuracy: {test_acc:.3f}")

### *From Model to App: a Flask Demo*
The same PyTorch classifier, served as a web page: see `flask-demo/` and `session7-demo - Flask PyTorch Neural Network Web App.md` in this folder.

---
## Task 1: Binary Classification with Breast Cancer Data  ⏱ 25 min · in pairs

Build a binary classification model to predict cancer malignancy.

- **load:** `sklearn.datasets.load_breast_cancer`
- **preprocess:** `StandardScaler`, then a train/test split
- **build:** ≥2 hidden layers (e.g. 128, 64 units, ReLU); 1 output neuron, **sigmoid** activation
- **train:** binary cross-entropy loss, Adam optimiser, 20 epochs, batch size 16
- **evaluate:** accuracy and a confusion matrix on the test set, and plot training vs. validation accuracy

If TensorFlow won't install, do this whole task in PyTorch (the stretch cell below).

### Hints
Try the task first. Open one hint at a time if you get stuck.

<details><summary><b>Hint 1:</b> split before you scale</summary>

Split first with `train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)`. Then `scaler.fit_transform(X_train)` and `scaler.transform(X_test)`: the scaler only ever learns from the training data.

</details>

<details><summary><b>Hint 2:</b> which output layer and loss?</summary>

Look back at *Choosing Your Output Layer*: binary classification means `Dense(1, activation="sigmoid")` with `loss="binary_crossentropy"`. Add `metrics=["accuracy"]` so it's reported during training.

</details>

<details><summary><b>Hint 3:</b> getting validation curves</summary>

Pass `validation_split=0.2` to `fit()`. Then `history.history["accuracy"]` and `history.history["val_accuracy"]` are the two lines to plot.

</details>

<details><summary><b>Hint 4:</b> predictions are probabilities, not classes</summary>

`model.predict(X_test)` returns values between 0 and 1. Turn them into 0/1 labels with `(probs > 0.5).astype(int).ravel()` before passing them to `accuracy_score` or `confusion_matrix`.

</details>

<details><summary><b>Hint 5:</b> PyTorch stretch</summary>

Leave the sigmoid off the last layer and use `nn.BCEWithLogitsLoss()`, which applies it for you. The labels must be floats in a column: `torch.tensor(y_train, dtype=torch.float32).view(-1, 1)`. To predict, use `torch.sigmoid(model(X)) > 0.5`.

</details>

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

data = load_breast_cancer()
X, y = data.data, data.target
print(X.shape, "| classes:", data.target_names)

# TODO 1: train/test split, then StandardScaler (fit on train, transform both)

# TODO 2: build a Sequential model: Input -> Dense(128, relu) -> Dense(64, relu) -> Dense(1, sigmoid)

# TODO 3: compile with loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"]

# TODO 4: fit for 20 epochs, batch_size=16, with validation_split=0.2

# TODO 5: plot accuracy and val_accuracy from history.history

# TODO 6: evaluate on the test set -- accuracy_score and confusion_matrix

**Stretch:** rebuild the same model in PyTorch, and check the two give similar accuracy.

In [ ]:
# Stretch: the same model in PyTorch
# (reuses X_train, X_test, y_train, y_test from the cell above -- scaled)

# TODO 1: turn the arrays into tensors -- features as float32, labels as float32 in a column (.view(-1, 1))

# TODO 2: nn.Sequential with Linear(30, 128), ReLU, Linear(128, 64), ReLU, Linear(64, 1) -- no sigmoid

# TODO 3: loss_fn = nn.BCEWithLogitsLoss(), optimiser Adam, DataLoader with batch_size=16

# TODO 4: the training loop for 20 epochs (zero_grad, forward, loss, backward, step)

# TODO 5: predict with torch.sigmoid(...) > 0.5, then accuracy_score and confusion_matrix

---
# Part 3 – Building a Regressor (California Housing)

### Warm-up: *Regression MLP: load and prepare the data*

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

housing = fetch_california_housing()

X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

### Warm-up: *Creating the Regression MLP* (Keras)

In [ ]:
model = keras.models.Sequential([
    keras.Input(shape=X_train.shape[1:]),
    keras.layers.Dense(30, activation="relu"),
    keras.layers.Dense(1)
])
model.compile(
    loss="mean_squared_error",
    optimizer=keras.optimizers.SGD(learning_rate=1e-3)
)
history = model.fit(X_train, y_train,
                    epochs=20, validation_data=(X_valid, y_valid))
mse_test = model.evaluate(X_test, y_test)

X_new = X_test[:3]
y_pred = model.predict(X_new)

### Warm-up: *Checking Training Loss* (Keras)

In [ ]:
plt.plot(pd.DataFrame(history.history))
plt.grid(True)
plt.gca().set_ylim(0, 1)
plt.show()

print(y_pred)

### Warm-up: *The Same Regressor in PyTorch*

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
X_valid_t = torch.tensor(X_valid, dtype=torch.float32).to(device)
y_valid_t = torch.tensor(y_valid, dtype=torch.float32).view(-1, 1).to(device)

model = nn.Sequential(
    nn.Linear(X_train.shape[1], 30), nn.ReLU(),
    nn.Linear(30, 1),            # one number out, no activation
).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t),
                           batch_size=32, shuffle=True)

### Warm-up: *PyTorch: train and predict*

In [ ]:
for epoch in range(20):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1).to(device)
    mse_test = loss_fn(model(X_test_t), y_test_t)
    y_pred = model(X_train_t[:3].to(device))

print("Test MSE:", mse_test.item())

---
## Task 2: Regression with California Housing Data  ⏱ 25 min

Build a neural network to predict house prices using the California Housing dataset.

- **load:** `sklearn.datasets.fetch_california_housing`
- **preprocess:** `StandardScaler`, then an 80/20 train/test split
- **build:** ≥2 hidden layers (e.g. 64, 32 units, ReLU); 1 output neuron, **no activation**
- **train:** MSE loss, Adam optimiser, 50 epochs, batch size 32
- **evaluate:** MSE on the test set; plot predicted vs. actual prices

### Hints
Try the task first. Open one hint at a time if you get stuck.

<details><summary><b>Hint 1:</b> same preparation as the warm-up</summary>

Split with `test_size=0.2, random_state=42`, then fit the `StandardScaler` on the training set only and transform both. Scaling matters much more here than for images: the features are on very different scales.

</details>

<details><summary><b>Hint 2:</b> which output layer and loss?</summary>

Regression means `Dense(1)` with **no** activation. A ReLU would stop it predicting below 0; a sigmoid would cap it at 1. Use `loss="mse"`.

</details>

<details><summary><b>Hint 3:</b> 50 epochs is a lot of output</summary>

Pass `verbose=0` to `fit()` to keep the notebook tidy, then plot `history.history["loss"]` and `["val_loss"]` instead.

</details>

<details><summary><b>Hint 4:</b> the predicted-vs-actual plot</summary>

`model.predict(X_test).ravel()` gives a flat array. `plt.scatter(y_test, y_pred, alpha=0.3)`, then draw the perfect-prediction line with `plt.plot([0, 5], [0, 5], "r--")`. Points close to the line are good predictions.

</details>

<details><summary><b>Hint 5:</b> PyTorch stretch</summary>

The warm-up regressor is almost exactly what you need: change the layers to 64 and 32, use `torch.optim.Adam`, and train for 50 epochs.

</details>

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

housing = fetch_california_housing()
X, y = housing.data, housing.target    # target: median house value, in $100,000s

# TODO 1: split 80/20 into train/test, then scale with StandardScaler

# TODO 2: build a Sequential model: Input -> Dense(64, relu) -> Dense(32, relu) -> Dense(1)

# TODO 3: compile with loss="mse", optimizer="adam"

# TODO 4: fit for 50 epochs, batch_size=32, with validation_split=0.2

# TODO 5: plot loss and val_loss, then evaluate MSE on the test set

# TODO 6: scatter-plot predicted vs. actual prices on the test set, with a y = x line

**Stretch:** rebuild the same model in PyTorch, and compare its test MSE with the Keras version.

In [ ]:
# Stretch: the same model in PyTorch
# (reuses the scaled X_train, X_test, y_train, y_test from the cell above)

# TODO 1: tensors -- features float32; targets float32 in a column (.view(-1, 1))

# TODO 2: nn.Sequential: Linear(8, 64), ReLU, Linear(64, 32), ReLU, Linear(32, 1)

# TODO 3: nn.MSELoss(), torch.optim.Adam, DataLoader with batch_size=32

# TODO 4: train for 50 epochs

# TODO 5: test MSE -- compare with your Keras result

---
## Stretch: Multi-Class Classification with Fashion MNIST  (independent study)

Build your own Fashion MNIST classifier from scratch. It follows the Part 2 warm-ups closely, so it's a good check that you can build one unaided.

- **load:** Fashion MNIST (reload it: the Part 3 cells reused the names `X_train`, `X_test` and so on)
- **preprocess:** scale the pixel values to [0, 1]
- **build:** ≥2 hidden layers (e.g. 256, 128 units, ReLU); 10 outputs, **softmax**
- **train:** sparse categorical cross-entropy loss, Adam optimiser, 10 epochs, batch size 64
- **evaluate:** test accuracy, and a 3×3 grid of test images showing predicted vs. true labels

### Hints
Try the task first. Open one hint at a time if you get stuck.

<details><summary><b>Hint 1:</b> reloading the data</summary>

Copy the first Part 2 warm-up: `keras.datasets.fashion_mnist.load_data()`, then divide by `255.`. Keep a validation set, or use `validation_split=0.1` in `fit()`.

</details>

<details><summary><b>Hint 2:</b> the model</summary>

The same as the Part 2 warm-up with different layer sizes: `Input([28, 28])`, `Flatten()`, `Dense(256)`, `Dense(128)`, `Dense(10, activation="softmax")`.

</details>

<details><summary><b>Hint 3:</b> the 3×3 grid</summary>

Loop `for i in range(9):`, use `plt.subplot(3, 3, i + 1)`, `plt.imshow(X_test[i], cmap="binary")`, and set a title like `f"pred: {class_names[p]}\ntrue: {class_names[t]}"`. Colour the title red when they differ.

</details>

<details><summary><b>Hint 4:</b> PyTorch version</summary>

Copy *PyTorch: define the model* and *the training loop*, change the layer sizes to 256 and 128, and use `batch_size=64`.

</details>

In [ ]:
# TODO 1: reload Fashion MNIST and scale the pixels to [0, 1]

# TODO 2: build the model -- Input(28, 28) -> Flatten -> Dense(256, relu) -> Dense(128, relu) -> Dense(10, softmax)

# TODO 3: compile with sparse categorical cross-entropy and Adam; fit for 10 epochs, batch_size=64

# TODO 4: plot training and validation accuracy; is it over-fitting?

# TODO 5: evaluate test accuracy

# TODO 6: plot a 3x3 grid of test images, titled "pred: ... / true: ..."

In [ ]:
# Stretch: the same classifier in PyTorch

# TODO: copy "PyTorch: define the model" and "PyTorch: the training loop", change the layers to 256 and 128,
#       use batch_size=64, train for 10 epochs, then report test accuracy

---
# Reflection

Answer these in the markdown cell below:
1. Where exactly does `model.fit()` "hide" backpropagation? Point to the lines in the PyTorch training loop that do the same job.
2. Why does the classifier's output layer use **softmax**, while the regressor's output layer has **no activation**?
3. How did you check whether your models were over-fitting?
4. Which framework did you prefer, Keras or PyTorch, and why?

## My reflection

*Write here...*